In [ ]:
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
%matplotlib inline
%config InlineBackend.figure_format = "retina"
import shapely # map-rendering library
import numpy as np
import fiona # enging for reading more robust data files
import contextily as cx
import os

In [ ]:
# first ensure you are in the right directory!
# os.chdir("/users/username/MinGenResources-DFW/Parks")

In [ ]:
# Reading in counties data
counties = gpd.read_file("data/parks_data/Counties.geojson", engine="fiona")
# Reading in Parks data
parks = gpd.read_file("data/parks_data/Parks_(2024).geojson")
# Changing Data to Coordinate System used in the Open Street Map API
counties_projected = counties.to_crs("EPSG:4326")
parks_projected = parks.to_crs("EPSG:4326")

In [ ]:
# Gets the geometry for Dallas County alone
dallas_geometry = counties_projected.loc[counties_projected["COUNTY"] == "Dallas", "geometry"].iloc[0]
# For each park, find if it intersects with Dallas County
parks_projected["intersects"] = parks_projected.intersects(dallas_geometry) 

In [ ]:
# Shows which parks intersect with Dallas County
parks_projected.plot(column="intersects", categorical=True, legend=True);

In [ ]:
# Subsets data into new data frame, only parks that intersect with dallas county
dallas_county_parks = parks_projected.loc[parks_projected["intersects"] == True]
dallas_county_parks.plot(color = "darkolivegreen")

In [ ]:
# The dataset contains proposed and existing parks. We only want the ones that actually exist
# Check to see the different categories
print(dallas_county_parks["STATUS"].value_counts())
# Subset the data only for existing parks
dallas_county_parks = dallas_county_parks[dallas_county_parks["STATUS"] == "Existing"]
# Check that this was done correctly
dallas_county_parks["STATUS"].value_counts()

In [ ]:
# Want to save just the parks in dallas county
# dallas_county_parks.to_file("dallas_county_parks.shp")

In [ ]:
# calculate the centroid
#dallas_county_parks["centroids"] = dallas_county_parks.centroid

# check that it looks okay
# ax = dallas_county_parks.plot(color="darkolivegreen", edgecolor="black", figsize=(10, 10))
# ax = dallas_county_parks.set_geometry("centroids").plot(ax=ax, color="red", markersize=5)
# cx.add_basemap(ax)
# ax.set_title("Dallas County Parks with Centroids")
# ax.set_axis_off();

# Take only the columns that we need
#centroids = dallas_county_parks[["NAME", "CITY", "centroids"]]
# Export the data with centroids as a shapefile
#centroids.to_file('centroids.shp')

In [ ]:
# For mapping purposes, make a new dataframe and project to the CRS for mapping
dallas_county_parks_map = dallas_county_parks.to_crs("EPSG:3857")
# calculate centroids in this CRS
dallas_county_parks_map["centroid"] = dallas_county_parks_map.centroid

In [ ]:
ax = dallas_county_parks_map.plot(color="darkolivegreen", edgecolor="black", figsize=(10, 10))
ax = dallas_county_parks_map.set_geometry("centroid").plot(ax=ax, color="red", markersize=5)
#cx.add_basemap(ax)
ax.set_title("Dallas County Parks with Centroids")
ax.set_axis_off();

In [ ]:
# Want to test a specific case for the Trinity River Greenbelt (the big green area in the middle)
# Does the centroid fall within the area of the park itself?

# First, we want to the find the OBJECT ID of the greenbelt
dallas_county_parks_map[dallas_county_parks_map["NAME"] == "Trinity River Greenbelt"]

In [ ]:
ax = dallas_county_parks_map[dallas_county_parks_map["OBJECTID"] == 1919].plot(color="darkolivegreen")
ax = dallas_county_parks_map[dallas_county_parks_map["OBJECTID"] == 1919].set_geometry("centroid").plot(ax=ax, color="red", markersize=15)
cx.add_basemap(ax)
ax.set_title("Trinity River Greenbelt")
ax.set_axis_off();

In [ ]:
# Another way to visualize this is by assigning the centroids for the greenbelt in different colors

# To test, make a color column with everything red
dallas_county_parks_map["color"] = "red"
# Then make the Greenbelt rows a different color so that we can see it
dallas_county_parks_map.loc[1918, "color"] = "blue"
dallas_county_parks_map.loc[2604, "color"] = "pink"

In [ ]:
# We can see that the blue dot is outside of the actual park area, while the pink one is within
ax = dallas_county_parks_map.plot(color="darkolivegreen", edgecolor="black", figsize=(10, 10))
ax = dallas_county_parks_map.set_geometry("centroid").plot(ax=ax, color=dallas_county_parks_map["color"], markersize=5)
#cx.add_basemap(ax)
ax.set_title("Dallas County Parks with Centroids")
ax.set_axis_off();